# Vision Transformer Object Detection - Kaggle Training Notebook

This notebook trains a Vision Transformer for object detection on foggy weather images using the PL-RT-DETR approach.

**Dataset**: VOC 2012 Filtered (with synthetic fog)

**Training Stages**:
1. **Teacher Network**: Train on clean images
2. **Student Network**: Train on foggy images with perceptual loss (knowledge distillation)

**Checkpoints**: Saved every 2 epochs to `/kaggle/working/logs`

## 1. Setup Kaggle Environment and Clone Repository

In [ ]:
import os
import sys

# Clone the repository from GitHub
!git clone https://github.com/habibour/fogg_object_det_vision_trans.git
os.chdir('/kaggle/working/fogg_object_det_vision_trans')

# To pull updates for already cloned repo, uncomment and run:
# !git pull origin main

print("✅ Repository cloned successfully!")
print(f"Current directory: {os.getcwd()}")
!ls -lh

## 2. Install Dependencies

In [ ]:
# Install required packages
!pip install -q ultralytics>=8.0.0
!pip install -q opencv-python-headless
!pip install -q albumentations
!pip install -q pycocotools
!pip install -q tensorboard

print("✅ All dependencies installed!")

# Verify installations
import torch
import ultralytics
import cv2
print(f"\n📦 Versions:")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA Available: {torch.cuda.is_available()}")
print(f"   Ultralytics: {ultralytics.__version__}")
print(f"   OpenCV: {cv2.__version__}")

## 3. Setup Dataset Paths (Kaggle Input)

In [ ]:
# Kaggle dataset paths
KAGGLE_VOC_PREFIX = "/kaggle/input/datasets/mdhabibourrahman/voc-2012-filtered"

# Dataset structure
DATASET_ROOT = f"{KAGGLE_VOC_PREFIX}/voc_2012/processed"
PAIRS_JSON = f"{DATASET_ROOT}/VOC2012_paired/pairs.json"

# Working directories
OUTPUT_DIR = "/kaggle/working/logs"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("📁 Dataset Paths:")
print(f"   Dataset Root: {DATASET_ROOT}")
print(f"   Pairs JSON: {PAIRS_JSON}")
print(f"   Output Dir: {OUTPUT_DIR}")
print(f"   Checkpoint Dir: {CHECKPOINT_DIR}")

# Verify dataset exists
if os.path.exists(PAIRS_JSON):
    print("\n✅ Dataset found!")
    # Check dataset structure
    !ls -lh {DATASET_ROOT}/VOC2012_paired/
else:
    print("\n⚠️  Dataset not found! Please add the dataset to Kaggle kernel.")
    print("   Expected path: /kaggle/input/datasets/mdhabibourrahman/voc-2012-filtered")

## 4. Configure Training Parameters

In [ ]:
# Training configuration
config = {
    # Dataset
    'pairs_json': PAIRS_JSON,
    'dataset_root': DATASET_ROOT,
    
    # Model
    'model_name': 'rtdetr-l',  # RT-DETR Large
    'num_classes': 5,  # bicycle, bus, car, motorbike, person
    'img_size': 640,
    
    # Training
    'batch_size': 8,  # Adjust based on GPU memory
    'num_workers': 2,
    'device': 'cuda',
    
    # Teacher training (Stage 1)
    'teacher_epochs': 20,  # Reduced for Kaggle (paper uses 100)
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    
    # Student training (Stage 2)  
    'student_epochs': 20,  # Reduced for Kaggle (paper uses 100)
    'perceptual_weight': 1.0,
    
    # Checkpointing
    'save_interval': 2,  # Save checkpoint every 2 epochs
    'val_interval': 2,   # Validate every 2 epochs
    
    # Paths
    'output_dir': OUTPUT_DIR,
    'checkpoint_dir': CHECKPOINT_DIR,
}

print("⚙️  Training Configuration:")
for key, value in config.items():
    print(f"   {key}: {value}")

## 5. Import Libraries and Load Modules

In [ ]:
# Add project root to path
sys.path.append('/kaggle/working/fogg_object_det_vision_trans')

# Import project modules
from dataset_loader import RTDETRDataset, collate_fn, create_dataloaders
from perceptual_loss import CombinedLoss, PerceptualLoss
from train_pl_rtdetr import PLRTDETRTrainer
from evaluate import Evaluator

# Standard imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import numpy as np
from tqdm import tqdm
import json
from pathlib import Path
from datetime import datetime

# Ultralytics RT-DETR
from ultralytics import RTDETR

print("✅ All modules imported successfully!")
print(f"   Device: {torch.device(config['device'])}")
print(f"   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 6. Load Dataset and Create DataLoaders

In [ ]:
print("📦 Loading datasets...")

# Load pairs.json to check dataset info
with open(PAIRS_JSON, 'r') as f:
    pairs_data = json.load(f)
    
print(f"\n📊 Dataset Statistics:")
print(f"   Total pairs: {len(pairs_data['pairs'])}")
print(f"   Fog levels: {pairs_data['metadata']['fog_levels']}")
print(f"   Classes: {pairs_data['metadata']['classes']}")

# Create datasets for Teacher training (clean images only)
train_dataset_clean = RTDETRDataset(
    pairs_json_path=PAIRS_JSON,
    dataset_root=DATASET_ROOT,
    split='train',
    img_size=config['img_size'],
    use_foggy=False,
    return_both=False
)

val_dataset_clean = RTDETRDataset(
    pairs_json_path=PAIRS_JSON,
    dataset_root=DATASET_ROOT,
    split='val',
    img_size=config['img_size'],
    use_foggy=False,
    return_both=False
)

# Create datasets for Student training (paired: both clean and foggy)
train_dataset_paired = RTDETRDataset(
    pairs_json_path=PAIRS_JSON,
    dataset_root=DATASET_ROOT,
    split='train',
    img_size=config['img_size'],
    use_foggy=True,
    random_fog=True,
    return_both=True
)

val_dataset_foggy = RTDETRDataset(
    pairs_json_path=PAIRS_JSON,
    dataset_root=DATASET_ROOT,
    split='val',
    img_size=config['img_size'],
    use_foggy=True,
    fog_level='mid',
    return_both=True
)

# Create dataloaders
train_loader_clean = DataLoader(
    train_dataset_clean,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=config['num_workers'],
    collate_fn=collate_fn,
    pin_memory=True
)

val_loader_clean = DataLoader(
    val_dataset_clean,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config['num_workers'],
    collate_fn=collate_fn,
    pin_memory=True
)

train_loader_paired = DataLoader(
    train_dataset_paired,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=config['num_workers'],
    collate_fn=collate_fn,
    pin_memory=True
)

val_loader_foggy = DataLoader(
    val_dataset_foggy,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config['num_workers'],
    collate_fn=collate_fn,
    pin_memory=True
)

print(f"\n✅ Dataloaders created:")
print(f"   Train (clean): {len(train_loader_clean)} batches")
print(f"   Val (clean): {len(val_loader_clean)} batches")
print(f"   Train (paired): {len(train_loader_paired)} batches")
print(f"   Val (foggy): {len(val_loader_foggy)} batches")

## 7. Stage 1: Train Teacher Network (Clean Images)

Train the teacher model on clean images from VOC 2012 dataset.

In [ ]:
print("="*60)
print("STAGE 1: Training Teacher Network on Clean Images")
print("="*60)

# Initialize RT-DETR teacher model
print("\n🤖 Loading RT-DETR model...")
teacher_model = RTDETR('rtdetr-l.pt')  # Load pretrained RT-DETR-L

# Configure for our 5 classes
# Note: We'll fine-tune the pretrained model for VOC filtered classes

# Setup tensorboard writer
writer = SummaryWriter(log_dir=f"{OUTPUT_DIR}/teacher_logs")

# Initialize teacher with config using the trainer class
print("\n🚀 Initializing teacher trainer...")
trainer = PLRTDETRTrainer(config)

# Train teacher
print(f"\n📚 Starting teacher training for {config['teacher_epochs']} epochs...")
print(f"   Checkpoints saved every {config['save_interval']} epochs")
print(f"   Validation every {config['val_interval']} epochs\n")

trainer.train_teacher()

print("\n✅ Teacher training completed!")
print(f"   Best model saved to: {CHECKPOINT_DIR}/teacher_best.pth")

## 8. Stage 2: Train Student Network (Foggy Images with Perceptual Loss)

Train the student model on foggy images using knowledge distillation from the teacher.

In [ ]:
print("="*60)
print("STAGE 2: Training Student Network with Knowledge Distillation")
print("="*60)

# Load best teacher checkpoint
teacher_checkpoint_path = f"{CHECKPOINT_DIR}/teacher_best.pth"
if os.path.exists(teacher_checkpoint_path):
    print(f"\n✅ Loading best teacher model from: {teacher_checkpoint_path}")
else:
    print(f"\n⚠️  Teacher checkpoint not found at: {teacher_checkpoint_path}")
    print("   Using current teacher model state")

# Setup student tensorboard writer
student_writer = SummaryWriter(log_dir=f"{OUTPUT_DIR}/student_logs")

# Train student with perceptual loss
print(f"\n📚 Starting student training for {config['student_epochs']} epochs...")
print(f"   Using perceptual loss for knowledge distillation")
print(f"   Perceptual weight: {config['perceptual_weight']}")
print(f"   Checkpoints saved every {config['save_interval']} epochs")
print(f"   Validation every {config['val_interval']} epochs\n")

trainer.train_student()

print("\n✅ Student training completed!")
print(f"   Best model saved to: {CHECKPOINT_DIR}/student_best.pth")

## 9. Evaluate Models on VOC Test Set

In [ ]:
print("="*60)
print("EVALUATION: Testing Models on VOC Dataset")
print("="*60)

# Load best models
print("\n📥 Loading best models...")
teacher_best = f"{CHECKPOINT_DIR}/teacher_best.pth"
student_best = f"{CHECKPOINT_DIR}/student_best.pth"

# Create test dataloaders for different conditions
test_conditions = {
    'clean': RTDETRDataset(
        pairs_json_path=PAIRS_JSON,
        dataset_root=DATASET_ROOT,
        split='val',  # Using validation set as test
        img_size=config['img_size'],
        use_foggy=False,
        return_both=False
    ),
    'fog_low': RTDETRDataset(
        pairs_json_path=PAIRS_JSON,
        dataset_root=DATASET_ROOT,
        split='val',
        img_size=config['img_size'],
        use_foggy=True,
        fog_level='low',
        return_both=False
    ),
    'fog_mid': RTDETRDataset(
        pairs_json_path=PAIRS_JSON,
        dataset_root=DATASET_ROOT,
        split='val',
        img_size=config['img_size'],
        use_foggy=True,
        fog_level='mid',
        return_both=False
    ),
    'fog_high': RTDETRDataset(
        pairs_json_path=PAIRS_JSON,
        dataset_root=DATASET_ROOT,
        split='val',
        img_size=config['img_size'],
        use_foggy=True,
        fog_level='high',
        return_both=False
    ),
}

# Evaluate on each condition
results = {}

print("\n📊 Evaluation Results:\n")
print(f"{'Condition':<15} {'Teacher mAP':<15} {'Student mAP':<15}")
print("-" * 50)

for condition_name, dataset in test_conditions.items():
    test_loader = DataLoader(
        dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=config['num_workers'],
        collate_fn=collate_fn
    )
    
    # Initialize evaluators
    teacher_evaluator = Evaluator(trainer.teacher, device=config['device'])
    student_evaluator = Evaluator(trainer.student, device=config['device'])
    
    # Evaluate
    teacher_metrics = teacher_evaluator.evaluate_on_dataset(test_loader)
    student_metrics = student_evaluator.evaluate_on_dataset(test_loader)
    
    results[condition_name] = {
        'teacher': teacher_metrics,
        'student': student_metrics
    }
    
    print(f"{condition_name:<15} {teacher_metrics['mAP']:<15.4f} {student_metrics['mAP']:<15.4f}")

print("\n✅ Evaluation completed!")

# Save results
results_path = f"{OUTPUT_DIR}/evaluation_results.json"
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
    
print(f"   Results saved to: {results_path}")

## 10. Visualize Training Metrics and Predictions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)

# Plot evaluation results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Teacher vs Student mAP comparison
conditions = list(results.keys())
teacher_maps = [results[c]['teacher']['mAP'] for c in conditions]
student_maps = [results[c]['student']['mAP'] for c in conditions]

x = np.arange(len(conditions))
width = 0.35

ax1.bar(x - width/2, teacher_maps, width, label='Teacher', color='steelblue')
ax1.bar(x + width/2, student_maps, width, label='Student', color='coral')
ax1.set_xlabel('Test Condition', fontsize=12)
ax1.set_ylabel('mAP', fontsize=12)
ax1.set_title('Model Performance Across Different Fog Levels', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(conditions, rotation=45)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Student improvement over teacher
improvements = [(student_maps[i] - teacher_maps[i]) / max(teacher_maps[i], 0.001) * 100 
                for i in range(len(conditions))]
colors = ['green' if imp > 0 else 'red' for imp in improvements]

ax2.barh(conditions, improvements, color=colors, alpha=0.7)
ax2.set_xlabel('Improvement (%)', fontsize=12)
ax2.set_title('Student Performance Improvement Over Teacher', fontsize=14, fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/evaluation_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

print("✅ Evaluation plots saved to: evaluation_comparison.png")

In [ ]:
# Visualize sample predictions
print("\n📸 Visualizing sample predictions...")

# Load a few test samples
sample_loader = DataLoader(
    test_conditions['fog_mid'],
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

# Get one batch
sample_batch = next(iter(sample_loader))
sample_images = sample_batch['images']
sample_targets = sample_batch['targets']

# Make predictions with student model
trainer.student.eval()
with torch.no_grad():
    predictions = trainer.student(sample_images.to(config['device']))

# Plot samples
fig, axes = plt.subplots(2, 2, figsize=(16, 16))
axes = axes.flatten()

class_names = ['bicycle', 'bus', 'car', 'motorbike', 'person']
colors = plt.cm.rainbow(np.linspace(0, 1, len(class_names)))

for idx in range(min(4, len(sample_images))):
    ax = axes[idx]
    
    # Convert image tensor to numpy
    img = sample_images[idx].permute(1, 2, 0).cpu().numpy()
    img = (img - img.min()) / (img.max() - img.min())  # Normalize to [0, 1]
    
    ax.imshow(img)
    ax.set_title(f'Sample {idx + 1} - Mid Fog', fontsize=12, fontweight='bold')
    ax.axis('off')
    
    # Note: Actual bounding box drawing would require parsing RT-DETR predictions
    # This is a placeholder - implement based on RT-DETR output format
    
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sample_predictions.png", dpi=300, bbox_inches='tight')
plt.show()

print("✅ Sample predictions saved to: sample_predictions.png")

## 11. View Training Logs with TensorBoard

In [ ]:
# Load TensorBoard in Kaggle notebook
%load_ext tensorboard

# Launch TensorBoard
print("🔥 Launching TensorBoard...")
print(f"   Teacher logs: {OUTPUT_DIR}/teacher_logs")
print(f"   Student logs: {OUTPUT_DIR}/student_logs")

# View teacher training logs
%tensorboard --logdir {OUTPUT_DIR}/teacher_logs --port 6006

# To view student logs separately, uncomment:
# %tensorboard --logdir {OUTPUT_DIR}/student_logs --port 6007

## 12. List All Saved Checkpoints and Models

In [ ]:
print("="*60)
print("SAVED ARTIFACTS")
print("="*60)

# List all checkpoints
print("\n📦 Checkpoints:")
!ls -lh {CHECKPOINT_DIR}

print("\n📊 Output Files:")
!ls -lh {OUTPUT_DIR}/*.png {OUTPUT_DIR}/*.json 2>/dev/null || echo "No output files found"

# Print summary of all saved files
import glob
from pathlib import Path

checkpoint_files = list(Path(CHECKPOINT_DIR).glob('*.pth'))
output_files = list(Path(OUTPUT_DIR).glob('*.png')) + list(Path(OUTPUT_DIR).glob('*.json'))

print(f"\n📌 Summary:")
print(f"   Total checkpoints: {len(checkpoint_files)}")
print(f"   Total output files: {len(output_files)}")
print(f"\n   All files saved to: /kaggle/working/logs")
print(f"   These files are available to download from Kaggle notebook output")

# Create a summary file
summary = {
    'training_completed': datetime.now().isoformat(),
    'config': config,
    'evaluation_results': results,
    'checkpoints': [str(f.name) for f in checkpoint_files],
    'outputs': [str(f.name) for f in output_files]
}

summary_path = f"{OUTPUT_DIR}/training_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Training summary saved to: {summary_path}")

## 📝 Next Steps and Usage Instructions

### Download Trained Models
All checkpoints and outputs are saved in `/kaggle/working/logs/` and can be downloaded from Kaggle:
- **Best Teacher Model**: `checkpoints/teacher_best.pth`
- **Best Student Model**: `checkpoints/student_best.pth`
- **Intermediate Checkpoints**: `checkpoints/teacher_epoch_*.pth` and `checkpoints/student_epoch_*.pth`

### To Use Trained Models:
```python
# Load student model for inference
from ultralytics import RTDETR
import torch

model = RTDETR('rtdetr-l.pt')
checkpoint = torch.load('/kaggle/working/logs/checkpoints/student_best.pth')
model.model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Make predictions
results = model.predict('path/to/foggy/image.jpg')
```

### Update Code from GitHub:
To pull latest changes from the repository, run:
```python
!cd /kaggle/working/fogg_object_det_vision_trans && git pull origin main
```

### Resume Training:
To resume training from a checkpoint, modify the trainer initialization to load existing checkpoints.

### Key Results:
- Teacher model trained on clean images
- Student model trained with perceptual loss for foggy conditions
- Checkpoints saved every 2 epochs
- Evaluated on multiple fog levels (low, mid, high) and clean images